In [1]:
import pandas as pd
import os, json, time
from dotenv import load_dotenv
from openai import OpenAI
import textwrap

#import truststore
#truststore.inject_into_ssl()



def pretty_print(*args):
    text = " ".join(str(arg) for arg in args)
    try:
        print(textwrap.fill(text, width=80))
    except Exception as e:
        print(text)  # fallback to normal print if text is not a string

        

load_dotenv(r"C:\Users\arunk\OneDrive\Documents\GenAI\LLM\openai_key.env")

api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise ValueError(
        "OPENAI_API_KEY not found! "
        "Make sure you have a .env file with: OPENAI_API_KEY=sk-..."
    )

pretty_print("API key loaded successfully.")

MODEL = 'gpt-4o-mini'




client = OpenAI(api_key=api_key)
pretty_print("OpenAI client ready.")

API key loaded successfully.
OpenAI client ready.


In [2]:
products = [
    {"id": "P001", "name": "Wireless Bluetooth Headphones", "description": "Over-ear wireless headphones with active noise cancellation, 30-hour battery life, and foldable design. Features Bluetooth 5.3 and multipoint connectivity."},
    {"id": "P002", "name": "Ergonomic Office Chair", "description": "Adjustable lumbar support office chair with breathable mesh back, 4D armrests, seat depth adjustment, and 135-degree recline. Supports up to 300 lbs."},
    {"id": "P003", "name": "Stainless Steel Water Bottle", "description": "Double-wall vacuum insulated 1-liter water bottle. Keeps drinks cold 24 hours or hot 12 hours. BPA-free, leak-proof lid with carrying loop."},
    {"id": "P004", "name": "Mechanical Keyboard", "description": "Compact 75% layout mechanical keyboard with hot-swappable switches, RGB backlighting, gasket mount structure, and PBT double-shot keycaps. USB-C and wireless."},
    {"id": "P005", "name": "Portable Power Station", "description": "500Wh portable power station with LiFePO4 battery, 800W pure sine wave AC output, 100W USB-C PD charging, and solar panel input. Weighs 14 lbs."},
    {"id": "P006", "name": "Smart Doorbell Camera", "description": "2K HDR video doorbell with 180-degree field of view, two-way audio, night vision, and local storage via microSD. Works with Alexa and Google Home."},
    {"id": "P007", "name": "Air Purifier", "description": "HEPA 13 air purifier covering 1,200 sq ft. Auto mode with air quality sensor, whisper-quiet 24dB sleep mode, and filter replacement indicator."},
    {"id": "P008", "name": "Camping Hammock", "description": "Ultralight double camping hammock made from 70D ripstop nylon. Holds 500 lbs. Includes tree straps, carabiners, and stuff sack. Packs down to 1 lb."},
    {"id": "P009", "name": "Electric Standing Desk", "description": "Dual-motor electric standing desk with 60x30 inch bamboo top. Height range 25.6-51.2 inches, 4 memory presets, cable management tray, and anti-collision sensor."},
    {"id": "P010", "name": "Robot Vacuum Cleaner", "description": "LiDAR navigation robot vacuum with 5000Pa suction, auto-empty dock, mop attachment, no-go zones via app, and 180-minute runtime on hardwood and carpet."}
]

df = pd.DataFrame(products)
df

,id,name,description
0,P001,Wireless Bluetooth Headphones,Over-ear wireless headphones with active noise...
1,P002,Ergonomic Office Chair,Adjustable lumbar support office chair with br...
2,P003,Stainless Steel Water Bottle,Double-wall vacuum insulated 1-liter water bot...
3,P004,Mechanical Keyboard,Compact 75% layout mechanical keyboard with ho...
4,P005,Portable Power Station,500Wh portable power station with LiFePO4 batt...
5,P006,Smart Doorbell Camera,2K HDR video doorbell with 180-degree field of...
6,P007,Air Purifier,"HEPA 13 air purifier covering 1,200 sq ft. Aut..."
7,P008,Camping Hammock,Ultralight double camping hammock made from 70...
8,P009,Electric Standing Desk,Dual-motor electric standing desk with 60x30 i...
9,P010,Robot Vacuum Cleaner,LiDAR navigation robot vacuum with 5000Pa suct...


In [3]:
short_prompt = "You are an assistant that summarizes products for SEO and extracts key features. Return JSON with keys: summary (2 sentences) and features (list of 3-5 strings)."

In [4]:
naive_results = []

for product in products[:5]:  # start with 5 products
    start = time.time()
    
    response = client.responses.create(
        model=MODEL,
        instructions=short_prompt,
        input=f"Product: {product['name']}\nDescription: {product['description']}"
    )
    
    elapsed = time.time() - start
    
    usage = response.usage
    cached = usage.input_tokens_details.cached_tokens if usage.input_tokens_details else 0
    
    naive_results.append({
        "product": product["name"],
        "input_tokens": usage.input_tokens,
        "output_tokens": usage.output_tokens,
        "cached_tokens": cached,
        "latency_s": round(elapsed, 2),
        "output_preview": response.output_text[:120] + "..."
    })
    
    print(f"✅ {product['name']}: {elapsed:.2f}s | input={usage.input_tokens} | cached={cached} | output={usage.output_tokens}")

print(f"\n⏱️  Total time: {sum(r['latency_s'] for r in naive_results):.2f}s")

✅ Wireless Bluetooth Headphones: 8.30s | input=86 | cached=0 | output=113
✅ Ergonomic Office Chair: 3.85s | input=89 | cached=0 | output=101
✅ Stainless Steel Water Bottle: 4.34s | input=87 | cached=0 | output=112
✅ Mechanical Keyboard: 4.32s | input=87 | cached=0 | output=126
✅ Portable Power Station: 2.83s | input=96 | cached=0 | output=127

⏱️  Total time: 23.64s


In [5]:
long_system_prompt = """
You are an expert e-commerce product content specialist working for a major online marketplace.
Your job is to create SEO-optimized product summaries and extract structured feature data.

=== OUTPUT FORMAT ===
You MUST respond with valid JSON only. No markdown, no explanation, just the JSON object.
{
  "seo_summary": "A compelling 2-3 sentence summary optimized for search engines. Include the product name, primary benefit, and a call-to-action phrase. Use active voice throughout.",
  "features": ["Feature 1 as a concise bullet point", "Feature 2", ...],
  "category": "The most specific product category",
  "target_audience": "Who this product is best suited for",
  "keywords": ["seo", "keyword", "list"]
}

=== WRITING STYLE GUIDE ===
1. Always write in active voice. BAD: "The battery is designed to last 30 hours." GOOD: "Enjoy 30 hours of uninterrupted battery life."
2. Lead with the primary benefit, not the feature. BAD: "Has Bluetooth 5.3" GOOD: "Connect seamlessly to all your devices with Bluetooth 5.3"
3. Use power words: discover, transform, elevate, essential, premium, revolutionary, seamless
4. Avoid superlatives without evidence. Don't say "best" or "greatest" unless the product description provides comparative data.
5. Keep sentences between 15-25 words for readability.
6. Include at least one emotional trigger word per summary (comfort, confidence, peace of mind, freedom, etc.)
7. End the SEO summary with an implicit call-to-action ("Upgrade your...", "Transform your...", "Experience...")

=== FEATURE EXTRACTION RULES ===
1. Extract 3-5 features maximum. Prioritize by consumer impact.
2. Each feature should be a standalone phrase (not a full sentence). 5-12 words each.
3. Always include the primary spec (battery life, weight, capacity, etc.) with units.
4. Group related specs into a single feature when possible.
5. Order features from most important to least important for the target consumer.

=== CATEGORY TAXONOMY ===
Use the most specific category from this hierarchy:
- Electronics > Audio > Headphones > Over-Ear / In-Ear / On-Ear
- Electronics > Computers > Peripherals > Keyboards / Mice / Monitors
- Electronics > Smart Home > Security / Lighting / Climate / Cleaning
- Furniture > Office > Chairs / Desks / Storage
- Outdoor & Sports > Camping > Shelter / Sleep / Cook / Accessories
- Kitchen & Home > Drinkware / Cookware / Appliances
- Power & Energy > Portable Power / Solar / Batteries

=== KEYWORD GUIDELINES ===
1. Generate 5-8 SEO keywords per product.
2. Include both short-tail (1-2 words) and long-tail (3-5 words) keywords.
3. Always include the product type, primary feature, and target use case.
4. Avoid brand names unless provided in the product description.
5. Include common misspellings or alternative terms consumers might search for.

=== FEW-SHOT EXAMPLES ===

EXAMPLE INPUT:
Product: Running Shoes
Description: Lightweight mesh running shoes with responsive foam cushioning, reinforced heel counter, and reflective details. Available in sizes 6-14. Weighs 8.5 oz.

EXAMPLE OUTPUT:
{
  "seo_summary": "Hit the pavement with confidence in these ultralight running shoes featuring responsive foam cushioning that absorbs impact mile after mile. The breathable mesh upper and reinforced heel deliver the perfect balance of comfort and stability. Elevate your running experience today.",
  "features": ["Responsive foam cushioning for impact absorption", "Breathable lightweight mesh upper at 8.5 oz", "Reinforced heel counter for stability", "Reflective details for low-light visibility", "Available in sizes 6 through 14"],
  "category": "Outdoor & Sports > Running > Footwear",
  "target_audience": "Recreational and intermediate runners seeking lightweight, cushioned footwear",
  "keywords": ["running shoes", "lightweight running shoes", "cushioned running shoes", "mesh running shoes", "reflective running shoes", "responsive foam shoes", "8.5 oz running shoes"]
}

EXAMPLE INPUT:
Product: Yoga Mat
Description: 6mm thick non-slip yoga mat made from natural tree rubber with alignment markings. Dimensions 72x26 inches. Includes carrying strap. Free from PVC, latex, and silicone.

EXAMPLE OUTPUT:
{
  "seo_summary": "Transform your practice on this premium natural rubber yoga mat with built-in alignment markings that guide your poses with precision. The 6mm thick non-slip surface provides the perfect cushion-to-grip ratio for any flow style. Discover your best practice with this eco-friendly essential.",
  "features": ["6mm natural tree rubber with non-slip grip", "Built-in alignment markings for guided practice", "72x26 inch full-size dimensions", "Eco-friendly: PVC, latex, and silicone free", "Includes carrying strap for portability"],
  "category": "Outdoor & Sports > Yoga > Mats",
  "target_audience": "Yoga practitioners of all levels seeking an eco-conscious, non-slip mat",
  "keywords": ["yoga mat", "non-slip yoga mat", "natural rubber yoga mat", "alignment yoga mat", "eco-friendly yoga mat", "6mm yoga mat", "yoga mat with markings"]
}

=== IMPORTANT REMINDERS ===
- Output ONLY the JSON object. No preamble, no markdown code fences, no explanation.
- If the product description is vague, make reasonable inferences but do not fabricate specific numbers.
- Use American English spelling throughout.
- Ensure all JSON keys are exactly as shown in the format specification above.
"""

# Rough token count check (1 token ≈ 4 chars for English)
estimated_tokens = len(long_system_prompt) / 4
print(f"Estimated system prompt tokens: ~{int(estimated_tokens)}")
print(f"Meets 1024-token minimum: {'✅ Yes' if estimated_tokens >= 1024 else '❌ No — need to expand'}")

Estimated system prompt tokens: ~1344
Meets 1024-token minimum: ✅ Yes


In [6]:
cache_results = []

for i, product in enumerate(products[:5]):
    start = time.time()
    
    response = client.responses.create(
        model=MODEL,
        instructions=long_system_prompt,
        input=f"Product: {product['name']}\nDescription: {product['description']}"
    )
    
    elapsed = time.time() - start
    usage = response.usage
    cached = usage.input_tokens_details.cached_tokens if usage.input_tokens_details else 0
    
    cache_results.append({
        "request_num": i + 1,
        "product": product["name"],
        "input_tokens": usage.input_tokens,
        "cached_tokens": cached,
        "cache_hit_pct": f"{(cached / usage.input_tokens * 100):.1f}%" if usage.input_tokens > 0 else "0%",
        "output_tokens": usage.output_tokens,
        "latency_s": round(elapsed, 2)
    })
    
    print(f"Request {i+1}: {product['name'][:30]:30s} | latency={elapsed:.2f}s | cached={cached}/{usage.input_tokens} tokens")

print(f"\n⏱️  Total: {sum(r['latency_s'] for r in cache_results):.2f}s")

Request 1: Wireless Bluetooth Headphones  | latency=3.80s | cached=0/1250 tokens
Request 2: Ergonomic Office Chair         | latency=5.78s | cached=0/1253 tokens
Request 3: Stainless Steel Water Bottle   | latency=6.96s | cached=0/1251 tokens
Request 4: Mechanical Keyboard            | latency=5.69s | cached=0/1251 tokens
Request 5: Portable Power Station         | latency=5.20s | cached=1024/1260 tokens

⏱️  Total: 27.43s


Batching API

In [7]:
batch_requests = []

for product in products:
    request = {
        "custom_id": f"product-{product['id']}",
        "method": "POST",
        "url": "/v1/responses",
        "body": {
            "model": MODEL,
            "instructions": long_system_prompt,
            "input": f"Product: {product['name']}\nDescription: {product['description']}"
        }
    }
    batch_requests.append(request)

# Write to JSONL file
batch_file_path = "batch_input.jsonl"
with open(batch_file_path, "w") as f:
    for req in batch_requests:
        f.write(json.dumps(req) + "\n")

print(f"Created {batch_file_path} with {len(batch_requests)} requests")
print(f"File size: {os.path.getsize(batch_file_path):,} bytes")

Created batch_input.jsonl with 10 requests
File size: 59,160 bytes


In [8]:
# Let's peek at what one line looks like
with open(batch_file_path) as f:
    first_line = json.loads(f.readline())

# Show structure (truncate the long prompt for readability)
preview = first_line.copy()
preview["body"] = {**preview["body"], "instructions": preview["body"]["instructions"][:100] + "..."}
print(json.dumps(preview, indent=2))

{
  "custom_id": "product-P001",
  "method": "POST",
  "url": "/v1/responses",
  "body": {
    "model": "gpt-4o-mini",
    "instructions": "\nYou are an expert e-commerce product content specialist working for a major online marketplace.\nYou...",
    "input": "Product: Wireless Bluetooth Headphones\nDescription: Over-ear wireless headphones with active noise cancellation, 30-hour battery life, and foldable design. Features Bluetooth 5.3 and multipoint connectivity."
  }
}


In [9]:
# Upload the JSONL file
batch_file = client.files.create(
    file=open(batch_file_path, "rb"),
    purpose="batch"
)
print(f"Uploaded file: {batch_file.id}")
print(f"Status: {batch_file.status}")

Uploaded file: file-XxcYx8HnDcnfneSVHnCuDf
Status: processed


In [10]:
# Create the batch job
batch_job = client.batches.create(
    input_file_id=batch_file.id,
    endpoint="/v1/responses",
    completion_window="24h",
    metadata={
        "description": "Product enrichment demo — 10 products"
    }
)

print(f"Batch ID: {batch_job.id}")
print(f"Status: {batch_job.status}")
print(f"Created at: {batch_job.created_at}")

Batch ID: batch_69b5c063118881909aeaf74c72ade345
Status: validating
Created at: 1773518947


In [11]:
poll_start = time.time()

while True:
    job = client.batches.retrieve(batch_job.id)
    elapsed = time.time() - poll_start
    
    completed = job.request_counts.completed if job.request_counts else 0
    total = job.request_counts.total if job.request_counts else 0
    
    print(f"[{elapsed:6.1f}s] Status: {job.status:15s} | Progress: {completed}/{total}")
    
    if job.status in ["completed", "failed", "expired", "cancelled"]:
        break
    
    time.sleep(10)

print(f"\n✅ Batch finished in {elapsed:.1f}s with status: {job.status}")
if job.request_counts:
    print(f"   Completed: {job.request_counts.completed} | Failed: {job.request_counts.failed}")

[   0.9s] Status: in_progress     | Progress: 4/10
[  11.8s] Status: in_progress     | Progress: 4/10
[  22.2s] Status: in_progress     | Progress: 4/10
[  33.4s] Status: in_progress     | Progress: 4/10
[  43.7s] Status: in_progress     | Progress: 4/10
[  54.5s] Status: in_progress     | Progress: 4/10
[  65.3s] Status: in_progress     | Progress: 4/10
[  75.6s] Status: completed       | Progress: 10/10

✅ Batch finished in 75.6s with status: completed
   Completed: 10 | Failed: 0


In [12]:
if job.status == "completed" and job.output_file_id:
    # Download the output file
    output_content = client.files.content(job.output_file_id)
    output_text = output_content.text
    
    # Save locally
    with open("batch_output.jsonl", "w") as f:
        f.write(output_text)
    
    # Parse each line
    batch_results = []
    for line in output_text.strip().split("\n"):
        result = json.loads(line)
        batch_results.append(result)
    
    print(f"Downloaded {len(batch_results)} results")
else:
    print(f"Batch did not complete successfully. Status: {job.status}")
    if job.error_file_id:
        error_content = client.files.content(job.error_file_id)
        print("Errors:", error_content.text[:500])

Downloaded 10 results


In [14]:
if batch_results:
    sample = batch_results[0]
    print(f"Custom ID: {sample['custom_id']}")
    print(f"Status code: {sample['response']['status_code']}")
    
    # The response body contains the same structure as a Responses API call
    body = sample['response']['body']
    
    # Extract the output text from the response
    output_items = body.get('output', [])
    for item in output_items:
        if item.get('type') == 'message':
            for content in item.get('content', []):
                if content.get('type') == 'output_text':
                    print(f"\nOutput text (first 300 chars):\n{content['text'][:300]}...")
    
    # Check usage
    usage = body.get('usage', {})
    print(f"\nUsage: input={usage.get('input_tokens', 'N/A')}, output={usage.get('output_tokens', 'N/A')}")
    print(f"Cached: {usage.get('input_tokens_details', {}).get('cached_tokens', 'N/A')}")

Custom ID: product-P001
Status code: 200

Output text (first 300 chars):
{
  "seo_summary": "Experience seamless listening with these over-ear wireless Bluetooth headphones featuring active noise cancellation that immerses you in your favorite music. Enjoy up to 30 hours of uninterrupted playtime while the foldable design makes them perfect for travel. Transform your aud...

Usage: input=1250, output=184
Cached: 0
